<a href="https://colab.research.google.com/github/Alina-cyber1/Detection_Number_Platet_KSA/blob/main/%D0%94%D0%B0%D1%82%D0%B0%D1%81%D0%B5%D1%82_%D0%B0%D0%B2%D1%82%D0%BE%D0%BD%D0%BE%D0%BC%D0%B5%D1%80%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Подключаем Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Устанавливаем нужные библиотеки
!pip install ultralytics opencv-python seaborn -q
print("Библиотеки установлены!")

In [ ]:
import os

dataset_path = '/content/drive/MyDrive/Тестовое задание_автомобили/Датасет_автономера/number_plates'

yaml_content = f"""
path: {dataset_path}
train: train
val: test

names:
  0: '0'
  1: '1'
  2: '2'
  3: '3'
  4: '4'
  5: '5'
  6: '6'
  7: '7'
  8: '8'
  9: '9'
  10: 'A'
  11: 'B'
  12: 'C'
  13: 'D'
  14: 'E'
  15: 'F'
  16: 'G'
  17: 'H'
  18: 'I'
  19: 'J'
  20: 'K'
  21: 'L'
  22: 'M'
  23: 'N'
  24: 'O'
  25: 'P'
  26: 'Q'
  27: 'R'
  28: 'S'
  29: 'T'
  30: 'U'
  31: 'V'
  32: 'W'
  33: 'X'
  34: 'Y'
  35: 'Z'
"""

with open('/content/data.yaml', 'w') as f:
    f.write(yaml_content)
print("Файл data.yaml успешно создан")

In [ ]:
import glob
import matplotlib.pyplot as plt
from collections import Counter
import os

dataset_path = '/content/drive/MyDrive/Тестовое задание_автомобили/Датасет_автономера/number_plates'
train_path = f'{dataset_path}/train'

# Считаем файлы
txt_files = [f for f in os.listdir(train_path) if f.endswith('.txt')]
jpeg_files = [f for f in os.listdir(train_path) if f.endswith('.jpeg')]

# Строим график
label_files = glob.glob(f'{train_path}/*.txt')
class_counts = Counter()

for file in label_files:
    with open(file, 'r') as f:
        for line in f.readlines():
            if line.strip():
                class_id = int(float(line.split()[0]))
                class_counts[class_id] += 1

classes = sorted(class_counts.keys())
counts = [class_counts[c] for c in classes]

plt.figure(figsize=(15, 5))
plt.bar(classes, counts, color='skyblue')
plt.title('Распределение символов в Train выборке')
plt.xlabel('ID класса (0-9 цифры, 10+ буквы)')
plt.ylabel('Количество')
plt.xticks(classes)
plt.show()

# Статистика
print(f"Количество изображений (.jpeg): {len(jpeg_files)}")
print(f"Количество файлов разметки (.txt): {len(txt_files)}")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os
import random
import xml.etree.ElementTree as ET

dataset_path = '/content/drive/MyDrive/Тестовое задание_автомобили/Датасет_автономера/number_plates/train'
# Cлучайная картинка
img_files = [f for f in os.listdir(dataset_path) if f.endswith('.jpeg')]
img_name = random.choice(img_files)
xml_name = img_name.replace('.jpeg', '.xml')

# Читаем картинку
img_path = os.path.join(dataset_path, img_name)
image = cv2.imread(img_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Читаем XML (координаты уже в пикселях)
tree = ET.parse(os.path.join(dataset_path, xml_name))
root = tree.getroot()

for obj in root.findall('object'):
    name = obj.find('name').text
    bndbox = obj.find('bndbox')

    xmin = int(bndbox.find('xmin').text)
    ymin = int(bndbox.find('ymin').text)
    xmax = int(bndbox.find('xmax').text)
    ymax = int(bndbox.find('ymax').text)

    # Рисуем рамку и подписываем
    cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
    cv2.putText(image, name, (xmin, ymin-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

# Показываем
plt.figure(figsize=(10, 6))
plt.imshow(image)
plt.title(f"Проверка разметки через XML: {img_name}")
plt.axis('off')
plt.show()

In [ ]:
# Обучение
from ultralytics import YOLO

# Загружаем самую быструю модель (Nano)
model = YOLO('yolov8n.pt')

# Запускаем обучение
results = model.train(
    data='/content/data.yaml',
    epochs=30,        # 30 эпох для быстрого результата
    imgsz=640,        # размер, к которому приводим картинки
    batch=16,         # сколько картинок обрабатывать за раз
    patience=10,      # если 10 эпох нет улучшений, остановить
    project='KSA_Detection',
    name='train_run'
)

In [ ]:
# Визуализация предсказаний
import glob
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Загружаем лучшую модель
best_model = YOLO('/content/runs/detect/KSA_Detection/train_run/weights/best.pt')

# Берем 3 картинки из теста
test_images = glob.glob('/content/drive/MyDrive/Тестовое задание_автомобили/Датасет_автономера/number_plates/test/*.jpeg')[:3]
for img_path in test_images:
    results = best_model(img_path)
    res_plotted = results[0].plot()

    plt.figure(figsize=(10, 5))
    plt.imshow(cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()